## Building the exercise dataset

Events and discussion questions from the same module or vignette of a CISA manual are combined into one exercise. An exercise is what a user of the platform actually runs, so this table is what the recommendation model (Goal 3) will recommend.

**What changed compared to the previous version:**
- **The 20 general discussion questions of the Executives manual are no longer lost.** The manual says they are designed to be used with both vignettes, so they are now added to both Executives exercises (step 4). Previously they were dropped in the merge and not saved anywhere.
- **List columns are saved as readable text** (`"Data Exposure; Ransomware"`) instead of Python list syntax (`"['Data Exposure', 'Ransomware']"`), which other notebooks would read back as one long string.
- **A new column `incident_threat_categories`** lists only the real threats in an exercise, without *No Confirmed Incident*, so exercises can be filtered by threat type.
- **Extra checks** that every event and every question ends up in at least one exercise (step 6).
- Exercise IDs (EX-01 … EX-16) are **unchanged**.

#### 1. Loading the data
The classified CISA events and discussion questions are loaded.

In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data/processed")
if not data_dir.exists():
    data_dir = Path("data/processed")

events = pd.read_csv(data_dir / "cisa_classified_events.csv")
questions = pd.read_csv(data_dir / "cisa_discussion_questions.csv")

print("Events:", events.shape)
print("Questions:", questions.shape)

Events: (66, 10)
Questions: (430, 9)


#### 2. Examining exercise structure
The modules and vignettes are compared to see how events and questions are organized across the manuals.

In [2]:
print("Events by section:")
display(events.groupby(["manual_title", "section"]).size().reset_index(name="event_count"))

print("Questions by section:")
display(questions.groupby(["manual_title", "section"]).size().reset_index(name="question_count"))

Events by section:


,manual_title,section,event_count
0,Communications,Module 1,4
1,Communications,Module 2,4
2,Executives,Vignette 1: Whaling/Ransomware,1
3,Executives,Vignette 2: Insider Threat,1
4,Financial Services,Module 1,4
5,Financial Services,Module 2,4
6,Healthcare,Module 1,5
7,Healthcare,Module 2,6
8,Industrial Control Systems,Module 1,5
9,Industrial Control Systems,Module 2,4


Questions by section:


,manual_title,section,question_count
0,Communications,Module 1,23
1,Communications,Module 2,22
2,Executives,General Discussion Questions,20
3,Executives,Vignette 1: Whaling/Ransomware,12
4,Executives,Vignette 2: Insider Threat,17
5,Financial Services,Module 1,29
6,Financial Services,Module 2,26
7,Healthcare,Module 1,34
8,Healthcare,Module 2,27
9,Industrial Control Systems,Module 1,32


#### 3. Handling the general discussion questions
The Executives manual has 20 general discussion questions that do not belong to a single vignette.

*Why:* the manual states that they are designed to be used with both vignettes, so each one is copied to *Vignette 1* and *Vignette 2*. Without this step they have no events to match and drop out of the dataset.

In [3]:
general_mask = (
    (questions["manual_title"] == "Executives")
    & (questions["section"] == "General Discussion Questions")
)
general_questions = questions[general_mask]

executive_vignettes = events.loc[events["manual_title"] == "Executives", "section"].unique()

questions_for_exercises = pd.concat(
    [questions[~general_mask]]
    + [general_questions.assign(section=vignette) for vignette in executive_vignettes],
    ignore_index=True,
)

print("General questions:", len(general_questions))
print("Executives vignettes they are added to:", list(executive_vignettes))
print("Question rows used for exercises:", len(questions_for_exercises))

General questions: 20
Executives vignettes they are added to: ['Vignette 1: Whaling/Ransomware', 'Vignette 2: Insider Threat']
Question rows used for exercises: 450


#### 4. Creating the exercise dataset
Events and discussion questions belonging to the same module or vignette are combined into one exercise record.

In [4]:
real_threat = lambda cats: sorted(c for c in cats.unique() if c != "No Confirmed Incident")

exercise_events = events.groupby(["manual_title", "section"], sort=False).agg(
    event_count=("inject_id", "size"),
    event_ids=("inject_id", list),
    threat_categories=("threat_category", lambda x: sorted(x.unique())),
    incident_threat_categories=("threat_category", real_threat),
    impact_categories=("impact_category", lambda x: sorted(x.unique())),
    event_text=("event_description", lambda x: " ".join(x)),
).reset_index()

exercise_questions = questions_for_exercises.groupby(["manual_title", "section"], sort=False).agg(
    question_count=("question_id", "size"),
    question_ids=("question_id", list),
    question_text=("question_text", lambda x: " ".join(x)),
).reset_index()

exercises = exercise_events.merge(
    exercise_questions, on=["manual_title", "section"], how="left", validate="one_to_one"
)

exercises["exercise_id"] = [f"EX-{i:02d}" for i in range(1, len(exercises) + 1)]

print("Total exercises:", len(exercises))
display(exercises[["exercise_id", "manual_title", "section", "event_count",
                   "question_count", "incident_threat_categories", "impact_categories"]])

Total exercises: 16


,exercise_id,manual_title,section,event_count,question_count,incident_threat_categories,impact_categories
0,EX-01,Healthcare,Module 1,5,34,"[Phishing, System Disruption, Vulnerability]","[No Direct Impact, Operational Disruption, Sec..."
1,EX-02,Healthcare,Module 2,6,27,"[Data Exposure, Insider Threat / Unauthorized ...","[Data Exposure, Operational Disruption, Patien..."
2,EX-03,Vendor,Module 1,4,28,[Third-Party / Supply Chain],"[Data Exposure, No Direct Impact]"
3,EX-04,Vendor,Module 2 – 6 months later,4,21,"[Other Security Incident, System Disruption, T...","[Operational Disruption, Security Risk]"
4,EX-05,Ransomware,Module 1,4,27,"[Other Security Incident, Phishing, Vulnerabil...","[No Direct Impact, Security Risk]"
5,EX-06,Ransomware,Module 2,4,32,"[Data Exposure, Ransomware]","[Data Exposure, Operational Disruption, Reputa..."
6,EX-07,Insider Threat,Module 1,5,22,"[Data Exposure, Insider Threat / Unauthorized ...","[Data Exposure, Financial Cost, Personnel Cons..."
7,EX-08,Insider Threat,Module 2,7,31,"[Insider Threat / Unauthorized Access, Third-P...","[Data Loss, Financial Cost, Security Risk]"
8,EX-09,Executives,Vignette 1: Whaling/Ransomware,1,32,[Ransomware],[Operational Disruption]
9,EX-10,Executives,Vignette 2: Insider Threat,1,37,[Insider Threat / Unauthorized Access],[Data Exposure]


#### 5. Creating exercise descriptions
The scenario events and discussion questions are combined into one text field for each exercise.

In [5]:
exercises["exercise_text"] = exercises["event_text"].fillna("") + " " + exercises["question_text"].fillna("")

display(exercises[["exercise_id", "manual_title", "section", "exercise_text"]].head())

,exercise_id,manual_title,section,exercise_text
0,EX-01,Healthcare,Module 1,The Cybersecurity and Infrastructure Security ...
1,EX-02,Healthcare,Module 2,A system administrator discovers active accoun...
2,EX-03,Vendor,Module 1,CISA and the Federal Bureau of Investigation (...
3,EX-04,Vendor,Module 2 – 6 months later,Employees at your organization are unable to s...
4,EX-05,Ransomware,Module 1,The Cybersecurity and Infrastructure Security ...


#### 6. Checking the exercise dataset
The exercises are checked against the original data.

*Why:* every event and every question must end up in at least one exercise; otherwise data is silently lost (as happened before with the general questions).

In [6]:
linked_events = set(exercises["event_ids"].explode())
linked_questions = set(exercises["question_ids"].explode())

assert len(exercises) == 16
assert exercises["exercise_id"].is_unique
assert exercises["exercise_text"].notna().all()
assert exercises["question_count"].notna().all(), "Exercise without questions"
assert linked_events == set(events["inject_id"]), "Some events are not in any exercise"
assert linked_questions == set(questions["question_id"]), "Some questions are not in any exercise"

print("Exercises:", len(exercises))
print("Events linked:", len(linked_events), "/", len(events))
print("Questions linked:", len(linked_questions), "/", len(questions))
print("All checks passed.")

Exercises: 16
Events linked: 66 / 66
Questions linked: 430 / 430
All checks passed.


#### 7. Saving the exercise dataset
The exercise records are saved as a CSV file.

*Why:* list columns are joined with `"; "` before saving, so the CSV contains readable values instead of Python list syntax. To get a list back: `df["event_ids"].str.split("; ")`.

In [7]:
list_columns = ["event_ids", "threat_categories", "incident_threat_categories",
                "impact_categories", "question_ids"]

exercises_out = exercises.copy()
for col in list_columns:
    exercises_out[col] = exercises_out[col].apply("; ".join)

exercises_out.to_csv(data_dir / "cisa_exercises.csv", index=False, encoding="utf-8-sig")

print("Exercises saved:", len(exercises_out))
print("Columns:", list(exercises_out.columns))

Exercises saved: 16
Columns: ['manual_title', 'section', 'event_count', 'event_ids', 'threat_categories', 'incident_threat_categories', 'impact_categories', 'event_text', 'question_count', 'question_ids', 'question_text', 'exercise_id', 'exercise_text']
